# [프로젝트] 요소수 유통 주유소 재고 현황 데이터 분석

---

## 프로젝트 목표
---
- 요소수 유통 주유소 재고 현황 데이터를 살펴보고 유의미한 데이터를 불러옵니다.
- 데이터를 지도에 표출하여 요소수 재고 현황을 조회할 수 있는 서비스를 제작합니다.

## 프로젝트 목차
---
1. **데이터 불러오기:** csv 데이터를 불러온 후 Dataframe 구조를 확인합니다.
2. **데이터 전처리하기:** 유의미한 데이터를 제공하기 위해 데이터를 전처리합니다.
3. **데이터 분석하기:** 다양한 방법으로 데이터를 파헤치고 통계해봅니다.
4. **데이터 시각화하기:** 지도에 데이터를 표출하여 요소수 재고 현황 조회 서비스를 제공합니다.

## 프로젝트 개요
---
현재 사회적 문제로 대두되고 있는 요소수 문제를 극복하기 위해 공공데이터 포털에서 요소수 유통 주유소 재고현황을 제공하고 있습니다.

본 실습을 통해 요소수 재고 현황 데이터를 지도에 표출하여 유의미한 데이터를 제공하는 서비스를 구축해봅니다.

**데이터 출처:** https://www.data.go.kr/index.do

---

## 1. 데이터 불러오기

pandas Dataframe 형태로 요소수 재고 현황 데이터를 불러들입니다.

In [ ]:
# 필요한 라이브러리 불러오기
import numpy as np
import pandas as pd

# 데이터 불러오기
df = pd.read_csv("./Urea data.csv")
df

불러온 데이터를 살펴보면 200개의 요소수 유통 주유소에 대한 11개의 정보가 있습니다.

각 컬럼은 왼쪽부터 `주소`, `주유소 코드`, `재고 레벨`, `재고`, `위도`, `경도`, `영업시간`, `가격`, `업데이트 일시`, 그리고 `전화번호`에 대한 정보를 가지고 있습니다.

---

## 2. 데이터 전처리하기

다음으로 데이터에 결측치(missing data)가 있는지 확인하고, 데이터 정제 과정을 수행해 봅시다.

In [ ]:
df.info() # 데이터 구조 확인

데이터 구조를 살펴보면 `영업시간` 대한 정보에 결측치가 있는 것을 알 수 있습니다. 이는 `영업시간`에 대한 정보가 존재하지 않는 다는 것을 의미합니다.

비어있는 열에서는 유의미한 정보를 얻을 수 없기 때문에 `df.drop()`을 사용하여 해당 데이터를 삭제합니다.

In [ ]:
df = df.drop('openTime', axis=1)
df

In [ ]:
df.info()

이제 `영업시간` 데이터가 삭제된 것을 확인할 수 있습니다.

이번에는 이상치가 있는지 살펴보겠습니다. `inventory`와 `price` 두 변수에 대해 진행해봅시다.

In [ ]:
inven_mean = df['inventory'].mean()
print(f"재고량 평균은 {inven_mean:.2f}개") # 문자열 포맷팅

In [ ]:
# 재고량이 가장 적은 관측값들
df['inventory'].sort_values()[:5]

In [ ]:
# 재고량이 가장 많은 관측값들
df['inventory'].sort_values()[-5:]

음수의 값을 가진다거나, 터무니 없이 큰 값을 가지는 케이스는 우선 없는 것 같습니다.

In [ ]:
price_mean = df['price'].mean()
print(f"가격 평균은 {price_mean:.2f}원") # 문자열 포맷팅

In [ ]:
# 가격이 가장 낮은 관측값들
df['price'].sort_values()[:5]

In [ ]:
# 가격이 가장 높은 관측값들
df['price'].sort_values()[-5:]

요소수 가격은 1000원에서 3000원 사이에 꽤 고르게 분포하고 있는 것을 알 수 있습니다. <br>
이 경우에도 별다른 이상치는 없는 것으로 보이므로, 본격적인 분석 단계로 넘어가겠습니다.

---

## 3. 데이터 분석하기

`matplotlib`과 `seaborn` 라이브러리를 이용하여 데이터 통계를 해봅시다.

In [ ]:
# 필요한 라이브러리 불러오기
import matplotlib.pyplot as plt
import seaborn as sns

### 3-1. 지역별 요소수 유통 주유소 분포 알아보기

먼저 `전화번호` 정보를 활용하여 지역별 요소수 유통 주유소의 분포를 알아봅시다.

국내 지역번호 리스트를 생성하고 `전화번호` 데이터를 기반으로 각 지역의 요소수 유통 주유소의 수를 알아봅시다.

In [ ]:
# 국내 지역번호 리스트 생성
list_tel = ['02', '031', '032', '033', '041', '042', '043', '044', '051', '052', '053', '054', '055', '061', '062', '063', '064']
df_region_by_tel = pd.Series(np.zeros(len(list_tel), dtype=np.int64), index=list_tel)

# 지역별 요소수 유통 주유소 수 세기
for i in list_tel:
    df_region_by_tel[i] = df['tel'].str.startswith(i).sum()

df_region_by_tel

`전화번호` 정보를 살펴보았을 때 대부분의 요소수 유통 주유소는 수도권, 특히 경기도에 밀집되어 있으며 지방에서는 찾기 힘든 것을 확인할 수 있습니다.

추가적으로 주소 (`addr`)을 이용해서도 유사한 분석을 진행해보겠습니다.

In [ ]:
# 문자열을 입력받아 첫번째 단어를 추출하는 간단한 보조함수를 먼저 정의합니다.
def extract(string):
    return string.split()[0] # 띄어쓰기를 기준으로 'split'

In [ ]:
addrs = df['addr'].apply(extract)
addrs

In [ ]:
addrs_cnt = addrs.value_counts().sort_values(ascending=False)
addrs_cnt

In [ ]:
addrs_cnt_df = pd.DataFrame(addrs_cnt).rename(columns={'addr':'counts'})
addrs_cnt_df

`인천`과 `인천광역시`가 데이터 수집 과정에서의 오류로 중복 기재되었으므로, 손으로 직접 처리해줍니다. <br>
`경기도`와 `경기`도 비슷한 방법으로 진행해줍니다.

In [ ]:
# 두 row의 인덱스 값을 입력받아, 첫번째 인덱스로 병합하는 간단한 보조함수를 정의합니다.
def merge_rows(df, rowA, rowB):
    cnt = df.loc[rowB]
    df.loc[rowA] += cnt
    df = df.drop(rowB)
    return df

In [ ]:
addrs_cnt_df = merge_rows(addrs_cnt_df, '인천', '인천광역시')
addrs_cnt_df

In [ ]:
addrs_cnt_df = merge_rows(addrs_cnt_df, '경기', '경기도')
addrs_cnt_df

`평택시`의 경우도 경기도의 일부로 분류되는 것이 옳겠으나, 데이터 수집 과정에서 노이즈가 들어간 경우라고 볼 수 있겠습니다. <br>
분석 필요에 따라 이 부분도 위와 같이 처리해줄 수 있겠습니다.

---

### 3-2. 요소수 재고 레벨 분포 알아보기

전국의 요소수 `재고량`의 분포를 `histogram`을 통해 알아봅시다.

In [ ]:
fig, ax = plt.subplots()
ax.hist(df['inventory'], bins=30)
ax.set_xlabel('inventory')
ax.set_ylabel('count')

위의 `histogram`을 살펴보면 `재고량`이 왼쪽에 많이 분포되어 있는 것을 확인할 수 있습니다.

`재고량` 데이터를 `boxplot`으로도 확인해 보고, 평균 `재고량`도 계산해 봅시다.

In [ ]:
df.boxplot(column='inventory', grid=False) # boxplot 생성

inven_mean = df['inventory'].mean() # 평균값 계산
print(f"요소수 재고량의 평균은 {inven_mean}개입니다.") # 문자열 포맷팅

요소수의 평균 재고량은 `1371.315`입니다.

위의 두 그래프만 봤을 때는 요소수 재고가 부족한 것처럼 보입니다. 요소수 `재고 레벨`을 막대그래프로 그려보고 재고 현황을 좀 더 파헤쳐 봅시다.

In [ ]:
plt.figure(figsize=(5,5))
sns.set(style='darkgrid')
ax = sns.countplot(x='color', data=df, palette=['#8FBC8B', '#F0E68C', '#FA8072', '#C0C0C0'], order=['GREEN', 'YELLOW', 'RED', 'GRAY'])

`GREEN`은 여유, `YELLOW`는 보통, `RED`는 부족, `GRAY`는 품절은 의미합니다.

위의 그래프를 보았을 때는 요소수 재고가 여유있는 것을 알 수 있습니다. 실제로 `재고량`이 `1000` 이상이면 `GREEN`으로 분류되기 때문에 요소수 재고는 여유있는 것입니다.

---

### 3-3. 요소수 가격 분포 알아보기

앞서 사용한 방법과 마찬가지로 `histogram`을 통해 요소수 `가격`의 분포를 알아봅시다. 

In [ ]:
fig, ax = plt.subplots()
ax.hist(df['price'], bins=10, color='indigo')
ax.set_xlabel('price')
ax.set_ylabel('count')
ax.set_facecolor('white')

`가격` 데이터를 `boxplot`으로도 확인해 봅시다. 요소수의 평균 가격도 알아봅시다.

In [ ]:
df.boxplot(column='price', grid=False) # boxplot 생성

price_mean = df['price'].mean() # 평균값 계산
print(f"요소수 가격의 평균은 {price_mean:.1f}원입니다.") # 문자열 포맷팅

요소수의 평균 가격은 `1738.5`원인 것을 확인하였습니다.

---

### 3-4. 요소수 가격과 재고의 상관관계 알아보기

이번에는 `scatterplot`을 이용하여 주유소 별 요소수 가격과 재고량 사이에 상관관계가 있는지 알아보겠습니다.

In [ ]:
fig, ax = plt.subplots()
ax.scatter(df['price'], df['inventory'], color='darkgreen')
ax.set_xlabel('price')
ax.set_ylabel('inventory')
plt.show()

이번에는 `pandas`의 `corr` 함수를 이용해서 가격과 재고량 사이의 통계적 상관관계를 확인해보겠습니다.

In [ ]:
corr = df.corr(method="pearson", numeric_only=True)["inventory"][
    "price"
]  # 피어슨 상관계수
print(
    f"가격과 재고량 사이의 피어슨 상관계수는 {corr:.3f}입니다."
)  # 문자열 포맷팅

관점에 따라 다르겠지만, 충분히 유의미한 선형 상관관계를 보인다고 말하기는 어렵겠습니다.

---

### 3-5. 주유소 이름에서 패턴 찾아보기

지금까지 가격이나 재고량과 같이 분석이 상대적으로 쉬운 변량들을 주로 다뤄봤습니다. <br>
이번에는 주유소의 이름에서 패턴을 추출하는 조금 더 창의적인 분석을 시도해보겠습니다.

분석에 앞서 분석을 도와줄 보조함수를 두 가지 정의하겠습니다. <br>
여러 카테고리에 대해 유사한 분석을 진행할 때 이와 같이 관련 함수를 미리 정의해두면 수고를 줄일 수 있습니다.

In [ ]:
# 분석 키워드와 가격 차이를 입력값으로 받아, 전체 평균과 비교해주는 함수를 정의합니다.
def price_insight(keyword, diff):
    if diff > 0:
        print(f"{keyword} 주유소의 요소수 가격은 전체 평균에 비해 {diff:.2f}원 저렴합니다.")
    else:
        print(f"{keyword} 주유소의 요소수 가격은 전체 평균에 비해 {-diff:.2f}원 비쌉니다.")    

In [ ]:
# 분석 키워드와 재고량 차이를 입력값으로 받아, 전체 평균과 비교해주는 함수를 정의합니다.
def inven_insight(keyword, diff):
    if diff > 0:
        print(f"{keyword} 주유소의 요소수 재고량은 전체 평균에 비해 {diff:.2f}만큼 적습니다.")
    else:
        print(f"{keyword} 주유소의 요소수 재고량은 전체 평균에 비해 {-diff:.2f}만큼 많습니다.")    

먼저 휴게소에 속해있는 주유소들을 살펴보겠습니다.

In [ ]:
tdf = df[df.name.str.contains('휴게소')]
tdf

In [ ]:
price_diff = df.price.mean() - tdf.price.mean() # 전체 평균과의 요소수 가격 차이
price_insight("휴게소", price_diff)

In [ ]:
inven_diff = df.inventory.mean() - tdf.inventory.mean() # 전체 평균과의 요소수 가격 차이
inven_insight("휴게소", inven_diff)

일반화하기에는 관측값이 부족하긴 하지만, 휴게소의 주유소들의 경우 요소수 가격은 더 저렴하고 대신 재고량도 부족한 경향이 있는 것 같습니다.

이번에는 "셀프" 주유소들을 살펴보겠습니다.

In [ ]:
tdf = df[df.name.str.contains('셀프')]
print(f"셀프 주유소 관측값 수: {tdf.shape[0]}")
tdf.head()

In [ ]:
price_diff = df.price.mean() - tdf.price.mean() # 전체 평균과의 요소수 가격 차이
price_insight('셀프', price_diff)

셀프 주유소가 더 저렴할 것이라는 일반적인 인식과는 달리, "요소수 가격" 측면에서는 그런 추세가 관측되지 않는군요.

In [ ]:
inven_diff = df.inventory.mean() - tdf.inventory.mean() # 전체 평균과의 요소수 가격 차이
inven_insight('셀프', inven_diff)

가격이 더 비싸서인지는 모르겠지만, 재고량은 더 많은 편입니다.

마지막으로 이름에 "알뜰"이 들어간 주유소들이 실제로 더 알뜰할지 살펴보겠습니다.

In [ ]:
tdf = df[df.name.str.contains('알뜰')]
print(f"알뜰 주유소 관측값 수: {tdf.shape[0]}")
tdf.head()

In [ ]:
price_diff = df.price.mean() - tdf.price.mean() # 전체 평균과의 요소수 가격 차이
price_insight("알뜰", price_diff)

역시 관측값이 충분하지는 않지만, 이 데이터에 따르면 알뜰 주유소의 요소수 가격은 평균보다 저렴한 편입니다.

---

## 4. 데이터 시각화하기

### 4-1. 지역별 요소수 재고 현황 지도에 나타내기

folium 라이브러리를 이용하여 요소수 재고 현황을 지도에 표출합니다.

In [ ]:
# folium 라이브러리 불러오기
!pip install folium
import folium

In [ ]:
# 서울 중심지 중구를 가운데 좌표로 잡아 지도를 출력합니다 
map_osm = folium.Map(location=[37.557945, 126.99419], zoom_start=10)
map_osm

요소수 재고 현황을 확인하고자 하는 지역을 `target_region`으로 설정합니다.

본 실습에서는 경기도의 요소수 재고 현황을 알아보겠습니다.

In [ ]:
# 요소수 유통 주유소를 찾고 싶은 지역을 설정합니다.
target_region = '경기' # 변경 가능

주소에 `target_region`이 포함된 정보를 찾아 Dataframe 형태로 불러옵니다.

In [ ]:
df_target_region = df[df['addr'].str.contains(target_region)]
df_target_region

In [ ]:
df_target_region.shape # target_region이 포함된 관측값의 갯수

이제 원하는 지역의 요소수 재고 현황을 시각화 해봅시다.

1. 지도에 표출할 내용은 주유소 명칭, 주소, 재고량, 전화번호입니다.
2. `lat`, `lng` 정보를 이용해 지도 상의 좌표를 가져옵니다.
3. 재고에 따라 지도에 표출할 마커의 색상을 정합니다.
4. `CircleMarker`를 이용해 마커를 생성합니다.

In [ ]:
map_osm = folium.Map(location=[37.557945, 126.99419], zoom_start=9)

for i in df_target_region.values:
    iframe = "<b>    명칭 : </b>"+str(i[6])+"<br>"+"<b>     주소 : </b>"+i[0]+"<br>"+"<b> 재고량 : </b>"+str(i[3])+"<br>"+"<b> 전화번호 : </b>"+str(i[8])
    popup = folium.Popup(iframe, min_width=300, max_width=800)
    
    # 요소수 재고에 따라 지도에 나타낼 마커의 색상을 정합니다.
    color = i[2].upper()
    if color == 'GREEN':
        color_code = '#9ACD32'
    elif color == 'YELLOW':
        color_code = '#FFFF00'
    elif color ==  'RED':
        color_code = '#DC143C'
    elif color == 'GRAY':
        color_code = '#696969'
    
    # CircleMarker를 사용하여 원하는 지역에 마커를 생성합니다.
    marker = folium.CircleMarker([i[4], i[5]],          # 위치
                                 radius=5,              # 범위
                                 color=color_code,      # 색상
                                 fill=True,            # 마커 채우기
                                 fill_opacity=1.0,
                                 popup=popup)           # 팝업 설정
    marker.add_to(map_osm)
map_osm

데이터를 시각화해보니 요소수 재고는 경기도 여러 지역에 골고루 분포되어 있는 것을 확인할 수 있습니다. `GREEN`과 `YELLOW` 마커가 분산되어 있기 때문입니다.

하지만 일부 주유소는 요소수가 매진되었거나 부족한 것으로 보입니다.

유사한 분석을 **가격**에 대해 진행해보겠습니다.

In [ ]:
map_osm = folium.Map(location=[37.557945, 126.99419], zoom_start=9)

for i in df_target_region.values:
    iframe = "<b>    명칭 : </b>"+str(i[6])+"<br>"+"<b>     주소 : </b>"+i[0]+"<br>"+"<b> 재고량 : </b>"+str(i[3])+"<br>"+"<b> 전화번호 : </b>"+str(i[8])
    popup = folium.Popup(iframe, min_width=300, max_width=800)
    
    # 요소수 가격에 따라 지도에 나타낼 마커의 색상을 정합니다.
    price = i[7]
    if price <= 1500:
        color_code = '#f9d71c'
    elif price <= 2000:
        color_code = '#ff6600'
    else:
        color_code = '#ff0000'
    
    # CircleMarker를 사용하여 원하는 지역에 마커를 생성합니다.
    marker = folium.CircleMarker([i[4], i[5]],          # 위치
                                 radius=5,              # 범위
                                 color=color_code,      # 색상
                                 fill=True,            # 마커 채우기
                                 fill_opacity=1.0,
                                 popup=popup)           # 팝업 설정
    marker.add_to(map_osm)
map_osm

---

### 4-2. 재고별 요소수 유통 주유소 지도에 나타내기

다음으로 요소수 재고가 있는 주유소를 찾아봅시다.
재고량을 나타내는 `target_color`를 설정합니다.

In [ ]:
# 재고량을 충분으로 설정
target_color = 'GREEN'

In [ ]:
df_target_color = df[df['color'].str.contains(target_color)]
df_target_color

In [ ]:
# 전국의 요소수 주유소 재고를 확인하기 위해 중심지 좌표와 지도의 크기를 조절합니다.
map_osm = folium.Map(location=[36.3504, 127.3845], zoom_start=7)

for i in df_target_color.values:
    iframe = "<b>    명칭 : </b>"+str(i[6])+"<br>"+"<b>     주소 : </b>"+i[0]+"<br>"+"<b> 재고량 : </b>"+str(i[3])+"<br>"+"<b> 전화번호 : </b>"+str(i[8])
    popup = folium.Popup(iframe, min_width=300, max_width=800)
    
    # CircleMarker를 사용하여 원하는 지역에 마커를 생성합니다.
    marker = folium.CircleMarker([i[4], i[5]],          # 위치
                                 radius=5,              # 범위
                                 color='#9ACD32',       # 색상 (본 실습에서는 재고에 따라 색상을 바꾸지 않아도 괜찮습니다)
                                 fill=True,            # 마커 채우기
                                 fill_opacity=1.0,
                                 popup=popup)           # 팝업 설정
    marker.add_to(map_osm)
map_osm

지도를 보면 요소수 재고는 수도권에 많이 남아있은 것을 알 수 있습니다. 지방에서는 `marker`를 찾기 어렵습니다.

이는 위의 `전화번호` 데이터에서 알아봤듯이 수도권 밖의 지역에 요소수 주유소가 적다는 사실과도 관련이 있습니다.

---

<span style="color:rgb(120, 120, 120)">본 학습 자료를 포함한 사이트 내 모든 자료의 저작권은 엘리스에 있으며 외부로의 무단 복제, 배포 및 전송을 불허합니다.

Copyright @ elice all rights reserved</span>